# Time-Series Forecasting — Walk-Forward Validation, Baselines, and the Random Walk

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2027S_predictive_analytics_QM474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>QM47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2027S_predictive_analytics_QM474/blob/main/notebooks/nb16_time_series_forecasting_student.ipynb)

---

> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises**. You are expected to complete both exercises before submitting your notebook.

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain why a forecasting problem needs a **time-respecting split** and **walk-forward validation** — the one place in this course where the random 80/20 split does not apply.
2. Compare an **incumbent baseline** (Seasonal-Naive for employment, the random walk for the stock market) with exponential smoothing (ETS), seasonal ARIMA and lag-feature machine learning **on identical forecast origins**.
3. **Describe paired per-origin differences** honestly, and explain why this course does not put a k-fold interval or a verdict on them.
4. Build **forecast bands**, check their **calibration before their sharpness**, and open a locked test window exactly once.
5. Work with **log returns** and explain why a random walk is so hard to beat for a stock index.
6. Forecast the **range** of the S&P 500 — the skill the Course Case Competition rewards.

---

## Setup

Everything in this notebook ships with Google Colab — there is nothing to install. `statsmodels` supplies the classical forecasting models (exponential smoothing and ARIMA) and the autocorrelation plot; scikit-learn supplies `Ridge` for the lag-feature model; `yfinance` (also preinstalled in Colab) downloads the S&P 500 in Part B.

The setup cell also defines `describe_paired`, the small helper you will use every time two forecasters are compared on the same origins. It reports the mean gain, its spread, how many origins each side won, and the extremes — and deliberately nothing more. Section 6 explains why.

In [ ]:
# Setup — everything below ships with Google Colab (no installs needed)
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import acf
from statsmodels.graphics.tsaplots import plot_acf
from sklearn.linear_model import Ridge

warnings.filterwarnings("ignore")   # statsmodels prints convergence chatter at some origins
RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 6)
pd.set_option("display.precision", 3)


def describe_paired(gains, label):
    """Describe paired per-origin gains (positive = the challenger was better at that origin).

    Forecast origins share most of their training history and their forecast windows sit
    side by side in time, so this course DESCRIBES the gains (mean, spread, every origin)
    instead of putting an interval around them or declaring a verdict.
    """
    g = np.asarray(gains, dtype=float)
    return pd.Series({
        "origins": len(g),
        "mean gain": round(g.mean(), 3),
        "SD of gains": round(g.std(ddof=1), 3),
        "origins won": f"{int((g > 0).sum())} of {len(g)}",
        "largest gain": round(g.max(), 3),
        "largest loss": round(g.min(), 3),
    }, name=label)


print("Setup complete!")
print(f"RANDOM_SEED = {RANDOM_SEED}")

**Reading the output:** `Setup complete!` means every library loaded. The seed is the course's `474`; nothing in this notebook is random, but the seed keeps the convention.

---

## 1. Why This Matters: Two Forecasts, One Rule

Two people in the same state government need a forecast this month.

The **state workforce planner** presents next year's retail-employment outlook to the legislature:

> *"I need a defensible forecast of US retail employment for the next twelve months, with a range the legislature can read. Today we simply repeat last year's month-by-month numbers. If a model does better, show me — and show me it would have done better before, not just this once."*

The **treasury officer of the state pension fund** moves cash between bonds and stocks every month:

> *"Can the same toolkit tell me where the S&P 500 will be a month from now? And if it cannot, can it at least tell me how far the market could move, so I size the cash buffer correctly?"*

Both questions are forecasts: the rows are **days or months in order**, and the thing you predict lies in the future. That changes one rule you have used since nb09. There, you split the data once at random (80/20) and cross-validated on shuffled folds. Here, shuffling would let a model learn from March while it is "predicting" February — a leak from the future. So this notebook is the course's **one exception to the random split**: a **time-respecting train/test split** (the most recent stretch is locked) plus **walk-forward validation** on the training history.

Part A (Sections 2–7) answers the planner with a seasonal series. Part B (Sections 8–11) answers the treasury officer with the S&P 500 — and shows why the stock market is a very different animal. Part B is also your preparation for the **Course Case Competition**, a group forecasting competition on the S&P 500 that starts with nb18.

> **A question that often comes up here:** *"Isn't this just cross-validation with a different splitter?"* The arithmetic is the same — fit, predict, score, repeat — but the folds obey time. Every training window ends before its forecast window starts, every forecast window is the same length as the real job (12 months for the planner, up to 20 trading days for the treasury officer), and the windows move forward through history. A model that looks good this way has been asked, again and again, exactly the question the stakeholder will ask it next month.

---

# Part A — The Workforce Planner

## 2. Load the Data and Look at It

The data are monthly US employment counts (thousands of people) from the US Bureau of Labor Statistics, as packaged with the open textbook *Forecasting: Principles and Practice* (Hyndman & Athanasopoulos). The course stores a copy in its GitHub repository. You keep the **Retail Trade** series: 80 years of monthly history with a long upward trend, recessions, and a strong holiday-season pattern.

In [ ]:
# Monthly US retail employment (thousands): BLS data as packaged with the FPP3 textbook
DATA_URL = ("https://raw.githubusercontent.com/davi-moreira/"
            "2027S_predictive_analytics_QM474/main/notebooks/us_employment.csv")
us_employment = pd.read_csv(DATA_URL, parse_dates=["ds"])
retail = (us_employment.query('unique_id == "Retail Trade"')
          .set_index("ds")["y"].asfreq("MS"))          # MS = one row per month (month start)
retail.name = "retail_employment"

print(f"Months          : {len(retail):,}")
print(f"Range           : {retail.index[0]:%Y-%m} -> {retail.index[-1]:%Y-%m}")
print(f"Missing values  : {retail.isna().sum()}")
print(f"Last value      : {retail.iloc[-1]:,.0f} thousand employees ({retail.index[-1]:%b %Y})")

**Reading the output:** The series has **969 months, from January 1939 to September 2019, with no missing values**, and ends at **15,611 thousand** retail employees (about 15.6 million people). The `asfreq("MS")` call fixes the frequency at one row per month, which the forecasting models below rely on: a model that knows the data are monthly knows that "12 steps back" means "the same month last year".

---

### 2.1 Three views of the same series

A forecaster's first look answers three questions: is there a **trend**, is there a **seasonal pattern**, and are there **shocks**? One figure answers all three: the full history, the last ten years up close, and the average month-over-month change by calendar month.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].plot(retail.index, retail.values, lw=0.8)
axes[0].set_title("1939-2019: trend, recessions, seasonality")
axes[0].set_ylabel("Thousands of employees")

recent = retail.loc["2009":]
axes[1].plot(recent.index, recent.values, lw=1, marker=".", ms=3)
axes[1].set_title("2009-2019 up close")

changes = retail.diff().loc["1990":]
by_month = pd.DataFrame({"month": changes.index.month, "change": changes.values})
by_month.boxplot(column="change", by="month", ax=axes[2], grid=False)
axes[2].set_title("Month-over-month change by calendar month, 1990-2019")
axes[2].set_xlabel("Month")
axes[2].set_ylabel("Change (thousands)")
fig.suptitle("")
plt.tight_layout()
plt.show()

print("Average month-over-month change by calendar month, 1990-2019 (thousands):")
print(by_month.groupby("month")["change"].mean().round(0).to_string())

**Reading the output:** The left panel shows a long upward **trend** interrupted by **recessions** (the dips around 1980–82, 1990–91, 2001 and the deep one in 2008–09). The middle panel shows the **seasonal pattern** that dominates any single year: a spike every November–December. The right panel and the printed averages put numbers on it: since 1990, retail has added on average **130 thousand jobs in October, 367 thousand in November and 186 thousand in December**, then shed **629 thousand in January** and another 193 thousand in February. Any forecast that ignores the calendar will be badly wrong every winter — which is why Seasonal-Naive, which simply repeats last year's month, is a serious incumbent rather than a straw man.

---

## 3. The Structural Rule: Lock the Most Recent Year

The planner's job is a 12-month forecast, so the **locked test window is the last 12 months** of the data. Everything before it is the training history. Nothing in Sections 4–6 looks at the locked year; Section 7 opens it once, for the model the planner commits to.

Notice what is *not* here: no `train_test_split(..., shuffle=True)`, no stratification, no random seed doing the splitting. In a forecasting problem the split point is a **date**, chosen by the business question.

In [ ]:
TEST_MONTHS = 12        # the planner forecasts one year ahead, so one year is locked
y_train, y_test = retail.iloc[:-TEST_MONTHS], retail.iloc[-TEST_MONTHS:]

print(f"Train : {y_train.index[0]:%Y-%m} -> {y_train.index[-1]:%Y-%m}  ({len(y_train)} months)")
print(f"Test  : {y_test.index[0]:%Y-%m} -> {y_test.index[-1]:%Y-%m}  ({len(y_test)} months, LOCKED until Section 7)")

fig, ax = plt.subplots(figsize=(12, 4))
shown = y_train.loc["2000":]
ax.plot(shown.index, shown.values, label="Training history (shown from 2000)")
ax.axvspan(y_test.index[0], y_test.index[-1], color="red", alpha=0.15,
           label="Locked test window (values not plotted)")
ax.set_ylabel("Thousands of employees")
ax.legend(loc="upper left")
ax.set_title("Time-respecting split: the test window is the most recent year")
plt.tight_layout()
plt.show()

**Reading the output:** The training history runs from **January 1939 to September 2018 (957 months)**; the locked test window is **October 2018 – September 2019 (12 months)**. The plot shades the locked year without drawing its values — the point of a lock is that you do not look, even at a picture.

---

## 4. Walk-Forward Validation: Five Origins, One Year Apart

In nb08 you learned five-fold cross-validation. The time-series version keeps the number five but changes what a fold is. A **forecast origin** is a date on which you pretend to stand: you fit the model on everything up to that date and forecast the next 12 months, which you can score because they are still inside the training history. Then you step one year forward and do it again.

The five origins below end exactly where the training history ends, so the last one rehearses the real job — forecasting the 12 months that follow. This is the course's **five folds = five forecast origins** for a monthly series.

In [ ]:
HORIZON = 12     # each forecast covers the next 12 months
N_ORIGINS = 5    # five forecast origins = five folds, one year apart
origins = [len(y_train) - HORIZON * (N_ORIGINS - k) for k in range(N_ORIGINS)]

calendar = pd.DataFrame([
    {"origin": k,
     "training ends": f"{y_train.index[o - 1]:%Y-%m}",
     "training months": o,
     "forecast window": f"{y_train.index[o]:%Y-%m} -> {y_train.index[o + HORIZON - 1]:%Y-%m}"}
    for k, o in enumerate(origins, start=1)]).set_index("origin")
print(calendar.to_string())

fig, ax = plt.subplots(figsize=(12, 3.2))
start = y_train.index[-15 * 12]
for k, o in enumerate(origins, start=1):
    ax.plot([start, y_train.index[o - 1]], [k, k], color="#1f77b4", lw=7, solid_capstyle="butt",
            label="training history (starts in 1939)" if k == 1 else None)
    ax.plot([y_train.index[o], y_train.index[o + HORIZON - 1]], [k, k], color="#ff7f0e", lw=7,
            solid_capstyle="butt", label="12 forecast months" if k == 1 else None)
ax.set_yticks(range(1, N_ORIGINS + 1))
ax.set_yticklabels([f"origin {k}" for k in range(1, N_ORIGINS + 1)])
ax.invert_yaxis()
ax.legend(loc="lower left")
ax.set_title("Walk-forward validation: the history grows and the forecast window moves forward (last 15 years shown)")
plt.tight_layout()
plt.show()

**Reading the output:** The five origins stand at the end of **September 2013, 2014, 2015, 2016 and 2017**. At the first, the models train on 897 months and forecast October 2013 – September 2014; at the last, they train on 945 months and forecast October 2017 – September 2018, the final year of the training history. Each training window contains all the earlier ones and grows by 12 months per step, and the five forecast windows tile the last five years of the history without overlapping. In the diagram, blue is what a model may learn from and orange is what it must forecast; no orange month is ever inside its own blue bar.

> **A question that often comes up here:** *"Is there a library that does this for me?"* Yes — `sklearn.model_selection.TimeSeriesSplit` and sktime's `ExpandingWindowSplitter` both generate expanding training windows followed by a forecast window. The loop in this notebook is written out by hand so you can see that walk-forward validation is nothing more than a `for` loop over origins: fit on the past, forecast the next window, score it, move forward. Whatever tool you use, check the calendar it produces — the last forecast window should end where your training history ends.

---

## 5. The Planner's Criteria, Declared Before Any Score

Following the course's champion protocol, the planner writes the criteria down **before** computing a single error:

- **Decision:** the 12-month retail-employment forecast presented to the legislature.
- **Metric:** mean absolute error (MAE) over the 12 forecast months, in thousands of employees — the unit the legislature reads. (RMSE punishes large misses more; MAPE and MASE make series of different sizes comparable. With one series in employee units, MAE is the plain choice.)
- **Incumbent:** **Seasonal-Naive** — "repeat last year's value for each month," the method the planner uses today.
- **Challengers, in order of operational burden:** Naive (repeat the last month), Drift (extend the straight line from the first to the last month), **ETS** (Holt-Winters exponential smoothing: a damped trend plus a monthly pattern, both re-weighted toward recent months), **SARIMA** (a seasonal ARIMA that models this month's change from last month's and last year's), and **Ridge on lagged changes** — the lag-feature machine-learning model: a Ridge regression that predicts next month's change from the last 12 monthly changes and is rolled forward one month at a time.
- **Margins (a stipulated policy, reported as a band):** small **12,500** employees of MAE (about 0.08% of the 15.6-million retail workforce) — a planner who updates the method whenever it measurably helps; **base 25,000** (about 0.16%); large **50,000** (about 0.32%) — a planner whose method changes must be re-explained to the legislature.
- **How the comparison is read:** paired per-origin differences, **described** (Section 6 explains why there is no interval and no verdict). The **declared default is to keep the incumbent** unless the described evidence makes the case for a challenger and the planner accepts it.

Now — and only now — run every candidate on the identical five origins.

In [ ]:
def seasonal_naive(history, h):
    """Repeat the last 12 observed months (the planner's current method)."""
    last_year = history.values[-12:]
    return np.array([last_year[i % 12] for i in range(h)])


def naive(history, h):
    return np.repeat(history.values[-1], h)


def drift(history, h):
    slope = (history.values[-1] - history.values[0]) / (len(history) - 1)
    return history.values[-1] + slope * np.arange(1, h + 1)


def ets(history, h):
    """Holt-Winters exponential smoothing: damped additive trend + additive monthly pattern."""
    fit = ExponentialSmoothing(history, trend="add", damped_trend=True,
                               seasonal="add", seasonal_periods=12).fit()
    return fit.forecast(h).values


def sarima(history, h):
    """Seasonal ARIMA(1,1,1)(0,1,1)12: one regular and one seasonal difference."""
    fit = ARIMA(history, order=(1, 1, 1), seasonal_order=(0, 1, 1, 12)).fit()
    return fit.forecast(h).values


def ridge_lagged_changes(history, h, n_lags=12):
    """Lag-feature ML: Ridge predicts next month's CHANGE from the last 12 changes,
    then forecasts recursively (each prediction becomes a lag for the following month)."""
    chg = np.diff(history.values)
    X = np.column_stack([chg[n_lags - lag: len(chg) - lag] for lag in range(1, n_lags + 1)])
    model = Ridge(alpha=1.0).fit(X, chg[n_lags:])
    recent, level, path = list(chg), history.values[-1], []
    for _ in range(h):
        x_next = np.array([recent[-lag] for lag in range(1, n_lags + 1)]).reshape(1, -1)
        step = model.predict(x_next)[0]
        recent.append(step)
        level += step
        path.append(level)
    return np.array(path)


# Declared before scoring: the incumbent first, then the challengers in order of burden
candidates = {
    "Seasonal-Naive (incumbent)": seasonal_naive,
    "Naive": naive,
    "Drift": drift,
    "ETS (Holt-Winters)": ets,
    "SARIMA": sarima,
    "Ridge on lagged changes": ridge_lagged_changes,
}


def walk_forward_mae(forecaster, series, origins, h=HORIZON):
    """MAE at each origin: fit on everything before the origin, forecast h months, score them."""
    maes = []
    for o in origins:
        history, actual = series.iloc[:o], series.iloc[o:o + h].values
        maes.append(np.mean(np.abs(actual - forecaster(history, h))))
    return np.array(maes)


fold_mae = pd.DataFrame({name: walk_forward_mae(f, y_train, origins) for name, f in candidates.items()},
                        index=[f"origin {k}" for k in range(1, N_ORIGINS + 1)])
print("MAE at each origin (thousands of employees):")
print(fold_mae.T.round(1).to_string())
print("\nAcross the five origins:")
print(pd.DataFrame({"mean MAE": fold_mae.mean(),
                    "SD across origins": fold_mae.std(ddof=1)}).round(1).to_string())

**Reading the output:** Read the table row by row before reading the averages. **Seasonal-Naive's error shrinks from 302.6 thousand at the first origin to 40.7 at the last.** Repeating last year's level is exactly one year of growth behind, so its error is roughly the year's growth — large while retail employment was still climbing (2013–2016), small once hiring flattened (2017–2018). **Naive** (235.3 on average) and **Drift** (196.3) are worse than the incumbent: Naive ignores the seasonal swing entirely, and Drift extends an 80-year average slope that no longer describes the 2010s. The three models that learn both the trend and the seasonal pattern — **ETS (71.1), SARIMA (60.2) and Ridge on lagged changes (53.7)** — have much lower average errors, but notice the fourth origin (October 2016 – September 2017): all three did *worse* than Seasonal-Naive there (111.0, 137.6 and 164.5 against 98.9). The averages alone would hide that. The standard deviations across origins (from 18.8 to 110.6) are a reminder that one origin can look very different from the next.

---

## 6. Paired Per-Origin Differences — Described, Not Certified

Each challenger faced the **same five origins** as the incumbent, so the fair comparison subtracts origin by origin: the **gain** at an origin is the incumbent's MAE minus the challenger's MAE (positive means the challenger was better there). Pairing removes what both models share — a hard year is hard for everyone — and leaves only the difference between them. Start with the strongest-looking challenger, ETS. The grey bands mark the declared margins (±12.5, ±25 and ±50 thousand).

In [ ]:
MARGINS = {"small": 12.5, "base": 25.0, "large": 50.0}   # thousand employees of MAE (Section 5)
incumbent = "Seasonal-Naive (incumbent)"

gains_ets = fold_mae[incumbent] - fold_mae["ETS (Holt-Winters)"]   # > 0: ETS better at that origin
print("Gain of ETS over Seasonal-Naive at each origin (thousand employees of MAE):")
print(gains_ets.round(1).to_string())
print()
print(describe_paired(gains_ets, "ETS vs Seasonal-Naive").to_string())

fig, ax = plt.subplots(figsize=(10, 4))
for m in MARGINS.values():
    ax.axhspan(-m, m, color="grey", alpha=0.12)
ax.axhline(0, color="black", lw=1)
ax.plot(range(1, N_ORIGINS + 1), gains_ets.values, "o-", color="#2ca02c", ms=8)
ax.set_xticks(range(1, N_ORIGINS + 1))
ax.set_xticklabels(calendar["forecast window"], rotation=15)
ax.set_ylabel("Gain over Seasonal-Naive\n(thousand employees of MAE)")
ax.set_title("ETS vs Seasonal-Naive, origin by origin")
plt.tight_layout()
plt.show()

**Reading the output:** ETS beat Seasonal-Naive by **262.9, 190.4 and 147.2 thousand** in the three windows from October 2013 to September 2016, and lost by **12.0 and 18.8 thousand** in the two most recent windows. Summarized: a **mean gain of 113.9 thousand with an SD of 125.1**, three origins won out of five, the largest loss 18.8. The plot makes the shape visible at a glance: three points far above every margin band, then two points just below zero, inside the base band. The average is driven by the early years; the most recent years point the other way.

> **A question that often comes up here:** *"In nb08 we put a corrected interval around the paired gain and read it against the margin. Why not here?"* Because the correction you used there — the $\sqrt{1/k + 1/(k-1)}$ factor — was derived for k-fold cross-validation, where every fold trains on the same amount of data and the folds are exchangeable. Walk-forward origins are neither: each training window contains all the earlier ones, the windows have different lengths, and neighbouring forecast years share economic conditions. Nobody has validated that correction for this design, so the honest move is to show every origin, the mean and the spread, and let the stakeholder see the pattern rather than print a verdict the arithmetic cannot back.

> **A question that often comes up here:** *"Is there a formal test for forecasts, then?"* Yes — the **Diebold–Mariano test** (Diebold & Mariano, 1995) compares two forecasters through the series of their loss differences and uses a standard error that allows those differences to be correlated over time, which is exactly the problem here; Harvey, Leybourne and Newbold (1997) added a small-sample correction. It is the standard tool in forecasting research. With only five origins it has very little power, so it would almost always say "not enough evidence" — which is the same message the description above gives you in plain sight. Treat it as a "going further" tool for when you have many origins, such as the weekly origins of Part B.

**The planner's decision.** The description is clear about what happened and silent about whether it will continue. ETS's large gains all come from years when retail employment was still growing — exactly when repeating last year's level falls a year of growth behind — and the two most recent years, when growth stalled, favor the incumbent by small amounts. The planner declared in advance that the default is to keep the incumbent unless the evidence makes the case, and five nested origins that split by economic regime do not make it. So the planner keeps Seasonal-Naive for this year's forecast and writes the champion statement:

> **Champion.** We keep **Seasonal-Naive** for the legislature's 12-month retail-employment forecast. **Evidence.** On the same five walk-forward origins, ETS had a mean gain of 113.9 thousand employees of MAE (SD 125.1), winning the three growth years by 147–263 thousand and losing the two most recent years by 12 and 19 thousand; we describe these differences rather than certify them, because the origins are nested in time. **Checks.** Section 7's band and the locked year. **What would change our mind.** ETS is the candidate for the next evaluation; we revisit if retail employment returns to steady growth, when the incumbent's one-year lag costs the most, or once enough origins accumulate for a formal comparison.

This is a judgment made under a declared policy, not a statistical verdict. A planner who had written down beforehand that "a mean gain above the base margin, winning most origins, is enough" would switch to ETS — and the point of the protocol is that this rule has to be written down **before** the scores are seen, not chosen after.

---

## 📝 PAUSE-AND-DO Exercise 1 — Describe the other two challengers (10 minutes)

**Task:** ETS is not the only strong challenger in the table. Compute the per-origin gains of **SARIMA** and of **Ridge on lagged changes** over the Seasonal-Naive incumbent, describe each with `describe_paired`, and count at how many origins each gain exceeds the **base margin** (25 thousand). Then write a two-sentence statement for the planner: what the gains show, and whether they change the decision above.

---

> 💡 **Gemini Prompt:** *"I have a DataFrame `fold_mae` with one row per forecast origin and one column per forecasting model (MAE in thousands of employees), a string `incumbent` naming the Seasonal-Naive column, a function `describe_paired(gains, label)`, a dict `MARGINS` with a `"base"` margin, and `N_ORIGINS = 5`. Build a DataFrame `gains` with two columns, `"SARIMA"` and `"Ridge on lagged changes"`, each equal to `fold_mae[incumbent]` minus that model's column. Print `gains` rounded to one decimal, then print `pd.concat` of `describe_paired` for both columns side by side, then, for each column, print how many origins have a gain larger than `MARGINS['base']`."*
>
> **After running, verify:**
> - [ ] `gains` has five rows (one per origin) and two columns
> - [ ] A positive gain means the challenger had the lower MAE at that origin
> - [ ] The summary shows the mean gain, the SD of the gains and the origins won for both challengers
> - [ ] You can say which origins each challenger lost, and by how much

In [ ]:
# YOUR SOLUTION CODE HERE
# Hints:
# gains = pd.DataFrame({"SARIMA": fold_mae[incumbent] - fold_mae["SARIMA"], ...})
# pd.concat([describe_paired(gains[c], c) for c in gains.columns], axis=1)
# (gains[c] > MARGINS["base"]).sum()

### YOUR ANSWER HERE:

**What the gains show (two sentences):**

**Does this change the planner's decision? Why or why not?**

---

## 7. Forecast Bands, Then the One-Time Opening of the Locked Year

The legislature asked for a range, not just a line. A forecast band is only useful if it is **calibrated first** — an 80% band should contain the truth about 80% of the time — and **sharp second** — among calibrated bands, the narrower one is more useful. The walk-forward origins let you check calibration before trusting any band.

Two bands are compared. The **Seasonal-Naive band** follows *Forecasting: Principles and Practice* (§5.5): a seasonal-naive forecast error is one year-over-year change, so within the first year the band has a constant width, set by the spread of the last ten years of year-over-year changes. The **SARIMA band** comes from the model itself and widens with the horizon, because uncertainty compounds month by month.

In [ ]:
Z80 = stats.norm.ppf(0.90)     # central 80% band: ±1.28 standard deviations


def seasonal_naive_sigma(history, years=10):
    """FPP3 §5.5: a seasonal-naive error is one year-over-year change; use their recent spread."""
    yoy = (history - history.shift(12)).dropna().iloc[-12 * years:]
    return yoy.std(ddof=1)


rows = []
for k, o in enumerate(origins, start=1):
    history, actual = y_train.iloc[:o], y_train.iloc[o:o + HORIZON].values
    sn, s = seasonal_naive(history, HORIZON), seasonal_naive_sigma(history)
    fc = ARIMA(history, order=(1, 1, 1), seasonal_order=(0, 1, 1, 12)).fit().get_forecast(HORIZON)
    lo, hi = fc.conf_int(alpha=0.20).values.T
    rows.append({"origin": k,
                 "SN share inside": np.mean(np.abs(actual - sn) <= Z80 * s),
                 "SN width": 2 * Z80 * s,
                 "SARIMA share inside": np.mean((actual >= lo) & (actual <= hi)),
                 "SARIMA width, month 1": hi[0] - lo[0],
                 "SARIMA width, month 12": hi[-1] - lo[-1]})
band_check = pd.DataFrame(rows).set_index("origin")
print("80% bands at the five walk-forward origins (widths in thousands of employees):")
print(band_check.round(2).to_string())
print(f"\nCoverage over all {N_ORIGINS * HORIZON} forecast months (target 80%): "
      f"Seasonal-Naive {band_check['SN share inside'].mean():.0%}, "
      f"SARIMA {band_check['SARIMA share inside'].mean():.0%}")

**Reading the output:** **Calibration first.** The Seasonal-Naive band contained **all 60 walk-forward months (100%)**; the SARIMA band contained **92%** (all 12 months at origins 1–3, 67% at origin 4 — the window where its point forecast missed — and 92% at origin 5). Both are above the 80% target, so neither band is overconfident here; both are conservative. **Sharpness second.** The Seasonal-Naive band is **803–847 thousand wide** at every month, because the last ten years of year-over-year changes include the 2008–09 recession swings. The SARIMA band starts at **about 98 thousand wide for the first month and widens to 511–520 thousand by month 12** — narrower everywhere, and widening with the horizon as uncertainty compounds. The honest summary for the planner: the incumbent's band is safe but very wide; that width is the price of a method that models neither trend nor recent shocks.

Now the ceremony. The planner has committed to a model (Section 6) and to a band (above). The locked year opens **once**, for that model only. The result is reported, not used to re-pick: if the test year flatters or embarrasses the model, that is information about the next forecast, not an invitation to swap in whichever challenger would have looked best.

In [ ]:
# THE CEREMONY: the locked year opens once, for the committed model only.
champion = "Seasonal-Naive (incumbent)"
forecast = seasonal_naive(y_train, TEST_MONTHS)
sigma = seasonal_naive_sigma(y_train)
lower, upper = forecast - Z80 * sigma, forecast + Z80 * sigma

test_mae = np.mean(np.abs(y_test.values - forecast))
cv_mean = fold_mae[champion].mean()
if abs(test_mae - cv_mean) <= MARGINS["base"]:
    position = "INSIDE"
elif test_mae < cv_mean:
    position = "BELOW (better than)"
else:
    position = "ABOVE (worse than)"
inside = np.mean((y_test.values >= lower) & (y_test.values <= upper))

print(f"Committed model            : {champion}")
print(f"Test MAE, 12 locked months : {test_mae:,.1f} thousand employees")
print(f"Walk-forward mean MAE      : {cv_mean:,.1f} -> the test result is {position} the walk-forward mean ± 25")
print(f"Months inside the 80% band : {inside:.0%} (band: forecast ± {Z80 * sigma:,.0f} thousand)")

fig, ax = plt.subplots(figsize=(12, 4.5))
hist = y_train.loc["2015":]
ax.plot(hist.index, hist.values, color="#1f77b4", label="Training history")
ax.plot(y_test.index, y_test.values, color="black", marker="o", ms=4, label="Actual (opened once)")
ax.plot(y_test.index, forecast, color="#d62728", ls="--", label="Seasonal-Naive forecast")
ax.fill_between(y_test.index, lower, upper, color="#d62728", alpha=0.15, label="80% band")
ax.set_ylabel("Thousands of employees")
ax.legend(loc="upper left")
ax.set_title("The locked year, opened once")
plt.tight_layout()
plt.show()

**Reading the output:** Seasonal-Naive's error on the locked year is **47.1 thousand employees** — far **below** its walk-forward mean of **185.0** (outside the ±25 margin on the good side) — and **all 12 months fell inside the 80% band** (forecast ± 417 thousand). Read this as information, not vindication: October 2018 – September 2019 was a flat year for retail employment, the condition in which repeating last year works best, just as the last walk-forward origin (40.7) suggested. The planner reports the test result and the walk-forward range together — in a growth year the error would look more like the 225–303 of 2013–2016 — and does **not** reopen the comparison because the test year happened to be kind.

> **A question that often comes up here:** *"Conformal prediction gave us intervals with a guarantee in earlier notebooks — why not use it here?"* The guarantee rests on exchangeability: any row could have been any other row. A time series breaks that on purpose — next month depends on this month — so the shuffled-data guarantee does not transfer. That is why this section checks the band on the walk-forward origins, the time-respecting stand-in for the guarantee.

---

# Part B — The Treasury Officer and the S&P 500

## 8. A Series That Behaves Differently

The treasury officer's question is about the **S&P 500**, the index of 500 large US companies. The function below downloads daily closing levels from Yahoo Finance through `yfinance` (preinstalled in Colab) for a **fixed window, January 2018 – October 2026**, so your numbers match this notebook. Index data belong to S&P Dow Jones Indices: you download them for your own coursework, and you do not republish them.

Prices are not what forecasters model. They model **log returns**:

$$r_t = \ln\left(\frac{P_t}{P_{t-1}}\right), \qquad \ln\left(\frac{P_{t+h}}{P_t}\right) = r_{t+1} + r_{t+2} + \dots + r_{t+h}.$$

A log return is close to the percentage change for small moves (a 1% rise gives $r \approx 0.00995$), it does not depend on the level of the index (a 1% day is a 1% day at 2,700 or at 7,000), and — the property you will use most — the return over $h$ days is simply the **sum** of the daily returns.

In [ ]:
def load_sp500(start="2018-01-01", end="2026-10-30", csv_path=None):
    """Daily S&P 500 closing levels, indexed by trading day.

    Default source: Yahoo Finance through yfinance (preinstalled in Colab). If Yahoo is
    unavailable, download the daily history yourself as a CSV with columns Date and Close
    and pass csv_path="your_file.csv".
    """
    if csv_path is not None:
        raw = pd.read_csv(csv_path, parse_dates=["Date"]).set_index("Date")["Close"]
    else:
        import yfinance as yf
        end_exclusive = (pd.Timestamp(end) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
        raw = yf.download("^GSPC", start=start, end=end_exclusive,
                          auto_adjust=False, progress=False)["Close"].squeeze()
    raw.index = pd.to_datetime(raw.index)
    if raw.index.tz is not None:
        raw.index = raw.index.tz_localize(None)
    close = raw.loc[start:end].dropna().astype(float)
    close.name = "close"
    return close


sp500 = load_sp500()
log_price = np.log(sp500)
returns = log_price.diff().dropna()          # daily log returns

print(f"Trading days     : {len(sp500):,} ({sp500.index[0]:%Y-%m-%d} -> {sp500.index[-1]:%Y-%m-%d})")
print(f"First / last     : {sp500.iloc[0]:,.2f} -> {sp500.iloc[-1]:,.2f}")
print(f"Mean daily return: {returns.mean() * 100:.3f}%   SD: {returns.std() * 100:.3f}%")
print(f"Up days          : {(returns > 0).mean():.1%}")
print(f"Largest daily fall / rise: {returns.min() * 100:.1f}% / {returns.max() * 100:.1f}%")

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(sp500.index, sp500.values, lw=0.9)
axes[0].set_yscale("log")
axes[0].set_title("S&P 500 closing level (log scale)")
axes[1].plot(returns.index, returns.values * 100, lw=0.5, color="#444444")
axes[1].set_title("Daily log return (%)")
plt.tight_layout()
plt.show()

**Reading the output:** The download returns **2,220 trading days from 2 January 2018 to 30 October 2026**; the index rose from **2,695.81 to 7,580.07**. The average daily log return is **0.047%** with a standard deviation of **1.205%** — the typical day's noise is about 25 times the typical day's drift — and **54.5%** of days were up. The largest one-day fall was **−12.8%** and the largest rise **+9.1%**. The return plot shows something the level plot hides: quiet stretches and turbulent stretches, with turbulence arriving in clusters rather than spread evenly.

---

## 9. What Can — and Cannot — Be Predicted

The autocorrelation function (ACF) from Part A's toolkit answers the key question directly. On the left, the ACF of the returns themselves: does today's **direction** say anything about tomorrow's? On the right, the ACF of the **absolute** returns: does today's **size of move** say anything about tomorrow's?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(returns, lags=20, zero=False, ax=axes[0])
axes[0].set_title("ACF of daily log returns (direction)")
plot_acf(returns.abs(), lags=20, zero=False, ax=axes[1])
axes[1].set_title("ACF of |daily log returns| (size of the move)")
for ax in axes:
    ax.set_xlabel("Lag (trading days)")
plt.tight_layout()
plt.show()

print("Autocorrelation, lags 1-5, returns   :", np.round(acf(returns, nlags=5)[1:], 3))
print("Autocorrelation, lags 1-5, |returns| :", np.round(acf(returns.abs(), nlags=5)[1:], 3))

**Reading the output:** On the left, the autocorrelations of the returns are small and change sign from lag to lag (**−0.144, +0.096, −0.031, −0.059, +0.053** for lags 1–5): a large move is partly reversed the next day on average, but these patterns are small, unstable in sign, and of the same order as the noise — today's direction says little about tomorrow's. On the right, the autocorrelations of the **absolute** returns are all clearly positive and decay slowly (**0.356, 0.396, 0.364, 0.355, 0.320**): a big move today, in either direction, makes a big move tomorrow more likely. This is **volatility clustering**, and it is the forecastable part of the stock market.

> **A question that often comes up here:** *"Why can't a good model find the pattern in direction?"* Because if a reliable pattern existed, traders would trade on it until it disappeared. That is the core of the efficient-markets argument (Fama, 1970): public information is quickly reflected in prices, so tomorrow's change is close to unpredictable from today's prices. Small, unstable patterns do appear in data — the lag-1 bar above is one — but they are of the same order as the noise, they come and go between periods, and they rarely survive trading costs. Volatility is different: calm periods and turbulent periods each last for a while (Engle, 1982), so the **size** of the next month's move is genuinely forecastable even when its direction is not.

---

## 10. Walk-Forward on the S&P 500: Can Anything Beat the Random Walk?

The incumbent here is the **random walk**: the best guess of the index a few days from now is today's level, so every forecast of the cumulative log return is zero. It is the stock-market version of the naive forecast. Three challengers, declared before scoring:

- **Drift** — the index keeps rising at its average daily pace of the last five years (1,260 trading days).
- **AR(1) on returns** — an ARIMA(1,0,0) fitted to the last three years of daily returns: tomorrow's return depends on today's, plus a mean.
- **Ridge on lagged returns** — lag-feature machine learning: a Ridge regression that predicts the $h$-day return from the last day's, last week's and last month's returns, trained only on windows whose outcome was already known at the origin.

The origins are **weekly**: 104 of them, five trading days apart, covering about two years and ending with the last origin whose 20-day future is known. At each origin every model forecasts the cumulative log return at **1, 5, 10, 15 and 20 trading days** — the next day, and one to four weeks ahead (a course "month"). The score at an origin is the MAE over those five horizons, in **percentage points**.

In [ ]:
HORIZONS = np.array([1, 5, 10, 15, 20])      # trading days ahead
lp = log_price.values
r = np.diff(lp)                              # r[i] = log return from day i to day i + 1
n = len(lp)
sp_origins = np.arange(n - 21 - 5 * 103, n - 20, 5)   # 104 weekly origins; the last has a full 20-day future


def cumulative_target(o):
    """Cumulative log return from the close on origin day o to each horizon."""
    return lp[o + HORIZONS] - lp[o]


def random_walk(past_r):
    return np.zeros(len(HORIZONS))


def drift_forecast(past_r):
    return past_r[-1260:].mean() * HORIZONS


def ar1_forecast(past_r):
    fit = ARIMA(past_r[-750:], order=(1, 0, 0), trend="c").fit()
    return np.cumsum(fit.forecast(int(HORIZONS.max())))[HORIZONS - 1]


def ridge_forecast(past_r):
    def feats(s):   # information available at the close of day s
        return np.array([past_r[s - 1], past_r[s - 5:s].sum(), past_r[s - 20:s].sum()])
    out = []
    for h in HORIZONS:
        starts = np.arange(21, len(past_r) - h + 1)[-750:]       # outcomes already known at the origin
        X = np.array([feats(s) for s in starts])
        y_h = np.array([past_r[s:s + h].sum() for s in starts])
        out.append(Ridge(alpha=1.0).fit(X, y_h).predict(feats(len(past_r)).reshape(1, -1))[0])
    return np.array(out)


sp_models = {"Random walk (incumbent)": random_walk, "Drift": drift_forecast,
             "AR(1) on returns": ar1_forecast, "Ridge on lagged returns": ridge_forecast}
errors = {name: [] for name in sp_models}
for o in sp_origins:
    past_r, actual = r[:o], cumulative_target(o)       # only returns up to the origin's close
    for name, f in sp_models.items():
        errors[name].append(np.mean(np.abs(actual - f(past_r))) * 100)
sp_err = pd.DataFrame(errors, index=log_price.index[sp_origins])

print(f"{len(sp_origins)} weekly origins: {sp_err.index[0]:%Y-%m-%d} -> {sp_err.index[-1]:%Y-%m-%d}")
print("Mean absolute error of the cumulative log return, horizons 1/5/10/15/20 (percentage points):")
print(sp_err.mean().round(3).to_string())

sp_gains = sp_err.drop(columns="Random walk (incumbent)").rsub(sp_err["Random walk (incumbent)"], axis=0)
print("\nPaired per-origin gains over the random walk (percentage points of MAE):")
print(pd.concat([describe_paired(sp_gains[c], c) for c in sp_gains.columns], axis=1).to_string())

fig, ax = plt.subplots(figsize=(12, 4.5))
for c in sp_gains.columns:
    ax.plot(sp_gains.index, sp_gains[c].cumsum(), label=c)
ax.axhline(0, color="black", lw=1)
ax.set_ylabel("Running total of gains\nover the random walk (pp of MAE)")
ax.set_title("Paired per-origin gains over the random walk, accumulated origin by origin")
ax.legend()
plt.tight_layout()
plt.show()

**Reading the output:** Over **104 weekly origins from 12 September 2024 to 2 October 2026**, the random walk's MAE is **2.004 percentage points**; Drift reaches **1.907**, AR(1) **1.930** and Ridge **1.940**. Every challenger has a positive mean gain — **+0.097 (Drift), +0.074 (AR(1)), +0.064 (Ridge)** — but each gain's SD is four to seven times its mean (0.383, 0.454, 0.463), and each challenger won only **64–65 of the 104 origins**. Those similar win counts are the clue: all three challengers forecast a positive return (Drift by construction; the AR(1) and the Ridge through their intercepts), and over these two years the S&P 500 mostly rose, so anything that leaned upward won a little more often than it lost. The running totals climb in bursts and fall back in sell-offs. Nothing here reads the market's *direction*; the gains are the market's average upward drift over this particular stretch. Under the declared default, the random walk stays the treasury officer's point forecast.

> **A question that often comes up here:** *"Drift won most of the origins — isn't that a real edge?"* It is a real description of these two years, and that is all. Drift is a bet that the market keeps rising at its five-year pace; in a rising market that bet pays, and in a falling market it loses by the same logic. The running-total plot shows the gains arriving in bursts rather than steadily, and the weekly origins overlap (each 20-day window shares 15 days with the next), so 104 origins carry far less independent information than the count suggests. The treasury officer's honest summary is: "over this stretch, assuming the market keeps rising helped a little; nothing we tried reads the market's direction."

---

## 11. Forecasting the Range Instead of the Direction

If the direction is close to unpredictable, the useful forecast is the **range**. Under a random walk with daily volatility $\sigma$, the $h$-day log return has standard deviation about $\sigma\sqrt{h}$ — uncertainty grows with the **square root** of the horizon, which is the S&P 500 version of the widening band you saw for SARIMA. A central 80% band is then $\pm 1.28\,\sigma\sqrt{h}$ around a zero forecast.

The cell below estimates $\sigma$ from the **last year** of daily returns at each origin and checks the 20-day band on the same 104 weekly origins: calibration first (how often was the realized 20-day return inside?), sharpness second (how wide was the band?).

In [ ]:
def rw_band(past_r, h, window=252):
    """Random-walk 80% band for the h-day log return, volatility from the last `window` days."""
    sd = past_r[-window:].std(ddof=1) * np.sqrt(h)
    return -Z80 * sd, Z80 * sd


inside, width, realized, upper_edge = [], [], [], []
for o in sp_origins:
    lo, hi = rw_band(r[:o], 20)
    y20 = lp[o + 20] - lp[o]
    inside.append(lo <= y20 <= hi)
    width.append((hi - lo) * 100)
    realized.append(y20 * 100)
    upper_edge.append(hi * 100)
print(f"Random walk + one-year volatility, 80% band for the 20-day return, {len(sp_origins)} origins:")
print(f"  share of realized returns inside the band : {np.mean(inside):.1%}  (target 80%)")
print(f"  average band width                        : {np.mean(width):.2f} percentage points")

dates = log_price.index[sp_origins]
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.fill_between(dates, -np.array(upper_edge), upper_edge, color="#1f77b4", alpha=0.2, label="80% band")
ax.scatter(dates, realized, s=12, c=np.where(inside, "#1f77b4", "#d62728"), label="realized 20-day return")
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("20-day log return (%)")
ax.set_title("Random-walk band for the next 20 trading days at each weekly origin (red = outside)")
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

**Reading the output:** The 80% band built from the last year's volatility contained **82.7%** of the realized 20-day returns across the 104 origins — close to the 80% target, slightly conservative — with an average width of **11.35 percentage points** (roughly ±5.7% around zero). In the plot, the red points (outside the band) cluster in the turbulent weeks, when the one-year volatility estimate was slow to react. That is the opening for Exercise 2.

---

## 📝 PAUSE-AND-DO Exercise 2 — Does a faster volatility estimate give a better band? (10 minutes)

**Task:** The band above weights every day of the last year equally. An **exponentially weighted** (EWMA) volatility gives recent days more weight, so it reacts faster when the market turns calm or turbulent. With weights $\lambda^{0}, \lambda^{1}, \lambda^{2}, \dots$ on the most recent squared returns ($\lambda = 0.94$, the RiskMetrics convention), the EWMA variance is the weighted average of the squared returns. For **h = 5 and h = 20**, compare the one-year band with the EWMA band on the 104 weekly origins: report the share of realized returns inside each 80% band and each band's average width. Which band would you give the treasury officer, judging calibration first and sharpness second?

---

> 💡 **Gemini Prompt:** *"I have a NumPy array `r` of daily log returns, a NumPy array `lp` of log prices, an array `sp_origins` of integer origin positions, and `Z80 = 1.2816`. For h in [5, 20] and for each origin `o`: (1) take `past = r[:o][-252:]`; (2) compute the one-year volatility `past.std(ddof=1)` and the EWMA volatility with weights `0.94 ** np.arange(252)[::-1]` as `np.sqrt(np.sum(w * past**2) / w.sum())`; (3) the realized return is `lp[o + h] - lp[o]`; (4) each band is `±Z80 * vol * np.sqrt(h)`. Collect, for each method and horizon, the share of origins whose realized return lies inside the band and the average band width in percentage points (width × 100). Print a small table with one row per (horizon, method)."*
>
> **After running, verify:**
> - [ ] The table has four rows: one-year and EWMA, for h = 5 and h = 20
> - [ ] Every coverage value is a share between 0 and 1, compared with the 0.80 target
> - [ ] Widths at h = 20 are roughly twice the widths at h = 5 (the square-root rule: $\sqrt{4} = 2$)

In [ ]:
# YOUR SOLUTION CODE HERE
# Hints:
# w = 0.94 ** np.arange(252)[::-1]          # the most recent day gets weight 1
# vol_ewma = np.sqrt(np.sum(w * past ** 2) / w.sum())
# inside = abs(lp[o + h] - lp[o]) <= Z80 * vol * np.sqrt(h)

### YOUR ANSWER HERE:

**Coverage and width of each band (h = 5 and h = 20):**

**Which band would you give the treasury officer, and why?**

---

## 12. Wrap-Up: Key Takeaways

### What We Learned Today:

1. **Forecasting is the one exception to the random split.** Lock the most recent stretch, and validate by walking forward through time: fit on the past, forecast the next window, score it, move on.
2. **Start from the incumbent.** Seasonal-Naive for a seasonal business series, the random walk for a stock index. Every challenger — ETS, SARIMA, lag-feature Ridge — must earn its place against it on identical origins.
3. **Describe paired per-origin differences.** Show every origin, the mean gain and its spread. With nested, time-ordered origins the k-fold correction does not apply, so the course describes instead of certifying; the Diebold–Mariano test is the formal tool when origins are plentiful.
4. **Bands: calibration first, sharpness second.** Check coverage on the walk-forward origins before trusting a band, and remember that uncertainty grows with the horizon.
5. **Markets are different.** Direction is close to unpredictable; volatility is not. The valuable stock-market forecast is an honest range.

### Remember:

> **"Beat the naive forecast on the same origins, or say plainly that you did not."**

**Next: nb18 — the Course Case Competition workflow.** You will pull the S&P 500 yourself, turn today's "range" idea into **quantile forecasts** of the index's return at 1, 5, 10, 15 and 20 trading days, score them with the competition's metric on weekly walk-forward origins against the random-walk benchmark, describe the paired per-origin differences exactly as you did here, and write a submission file that the competition's checker accepts. Everything you need from today carries over: the time-respecting split, the walk-forward loop, the incumbent, the honest description — and the lesson that the range is where the skill shows.

> **A question that often comes up here:** *"If nothing beats the random walk, how can a forecasting competition be fair?"* By scoring the whole forecast, not just its center. In the competition you submit a **range** — several quantiles of the S&P 500's return at each horizon — and the score rewards ranges that are both honest and tight. A team that reads the market's volatility well can beat a team that guesses the direction confidently, and a large share of the grade goes to the walk-forward method and the honest comparison with the random walk, which you control completely.

---

## 13. Participation Assignment Submission Instructions

### To Submit This Notebook:

1. **Complete both exercises**: Fill in both PAUSE-AND-DO exercise cells with your code and your answers
2. **Run All Cells**: Execute `Runtime → Run all` to ensure everything works
3. **Save a Copy**: `File → Save a copy in Drive or Download the .ipynb extension`
4. **Submit**: Upload your `.ipynb` file in the participation assignment you find in the course Brightspace page.

### Before Submitting, Check:

- [ ] All cells execute without errors
- [ ] All outputs are visible
- [ ] Both exercise responses are complete
- [ ] You can explain every line of code you wrote

### Next Step:

Complete the **Quiz** in Brightspace (auto-graded)

---

## Bibliography

- Diebold, F. X., & Mariano, R. S. (1995). Comparing predictive accuracy. *Journal of Business & Economic Statistics*, 13(3), 253–263.
- Engle, R. F. (1982). Autoregressive conditional heteroscedasticity with estimates of the variance of United Kingdom inflation. *Econometrica*, 50(4), 987–1007.
- Fama, E. F. (1970). Efficient capital markets: A review of theory and empirical work. *The Journal of Finance*, 25(2), 383–417.
- Harvey, D., Leybourne, S., & Newbold, P. (1997). Testing the equality of prediction mean squared errors. *International Journal of Forecasting*, 13(2), 281–291.
- Hyndman, R. J., & Athanasopoulos, G. (2021). *Forecasting: Principles and Practice* (3rd ed.). OTexts. https://otexts.com/fpp3/ — §5.5 (prediction intervals for the benchmark methods), §5.10 (time-series cross-validation), Chapters 8–9 (exponential smoothing and ARIMA).
- J.P. Morgan/Reuters (1996). *RiskMetrics — Technical Document* (4th ed.) — the EWMA volatility with $\lambda = 0.94$.
- statsmodels documentation: [ExponentialSmoothing](https://www.statsmodels.org/stable/generated/statsmodels.tsa.holtwinters.ExponentialSmoothing.html), [ARIMA](https://www.statsmodels.org/stable/generated/statsmodels.tsa.arima.model.ARIMA.html).

---

<center>

Thank you!

</center>